<a href="https://colab.research.google.com/github/e23395/Statistical-Learning-e23395/blob/main/Assignment%207d%3A%20Structural%20Health%20Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates**

##**Task 1: Prior Belief Boundaries**

**Analytical Calculation of Expected Value**

For a Beta-distributed random variable $\Theta \sim \text{Beta}(\alpha, \beta)$ with parameters $\alpha = 8$ and $\beta = 1.5$, the expectation is given by:

$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421 \text{ (or } 84.21\% \text{ efficiency)}$$

**Engineering Justification of Prior Choice**

*   Physical Support Matching: The Beta distribution has a bounded support over $(0, 1]$, which aligns with the physical boundaries of the stiffness factor $\theta$.

*   Skewed Belief: With $\alpha = 8 > \beta = 1.5$, the distribution is heavily skewed toward $\theta = 1.0$. This assigns high probability mass to healthy states (near $1.0$) while smoothly decaying toward zero as $\theta \to 0$, accurately modeling engineering confidence in a newly deployed or recently inspected structural component.



**Python Code for Plotly Visualization**

In [3]:
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
from scipy.stats import beta

# Set the plotly template for a cleaner look
pio.templates.default = "plotly_white"

# =============================================================================
# TASK 1: Prior Belief Boundaries
# =============================================================================

# Define the Beta distribution parameters
alpha = 8
beta_param = 1.5

# Generate the domain (restricted to physical bounds)
theta = np.linspace(0.01, 1.0, 1000)

# Compute the PDF values
pdf_values = beta.pdf(theta, alpha, beta_param)

# Calculate the expected value analytically
expected_value = alpha / (alpha + beta_param)
print(f"Expected prior stiffness efficiency E[Θ⁽⁰⁾] = {expected_value:.4f}")

# Calculate the mode of the Beta distribution
mode = (alpha - 1) / (alpha + beta_param - 2)
print(f"Mode of the prior distribution = {mode:.4f}")

# Create the Plotly figure
fig = go.Figure()

# Add the prior density trace
fig.add_trace(go.Scatter(
    x=theta,
    y=pdf_values,
    mode='lines',
    name=f'Beta({alpha}, {beta_param})',
    line=dict(color='#1f77b4', width=3),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))

# Add a vertical line at the expected value
fig.add_vline(
    x=expected_value,
    line_dash="dash",
    line_color="#ff7f0e",
    line_width=2,
    annotation_text=f"E[Θ] = {expected_value:.3f}",
    annotation_position="top right"
)

# Add a vertical line at the mode
fig.add_vline(
    x=mode,
    line_dash="dot",
    line_color="#2ca02c",
    line_width=2,
    annotation_text=f"Mode = {mode:.3f}",
    annotation_position="top left"
)

# Update layout
fig.update_layout(
    title=dict(
        text="Initial Prior Distribution: Beta(8, 1.5)",
        font=dict(size=20, family="Arial", color="black"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title=dict(
            text="θ (Remaining Stiffness Efficiency Factor)",
            font=dict(size=14, family="Arial")
        ),
        range=[0, 1.05],
        tickvals=np.arange(0, 1.1, 0.1),
        gridcolor='lightgray',
        gridwidth=0.5
    ),
    yaxis=dict(
        title=dict(
            text="Probability Density",
            font=dict(size=14, family="Arial")
        ),
        gridcolor='lightgray',
        gridwidth=0.5
    ),
    legend=dict(
        x=0.7,
        y=0.95,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='black',
        borderwidth=1
    ),
    hovermode='x',
    width=900,
    height=500
)

# Add annotations for interpretation
fig.add_annotation(
    x=0.85,
    y=0.5,
    text="High density near θ=1.0<br>indicating healthy structure",
    showarrow=True,
    arrowhead=2,
    arrowsize=1.5,
    arrowwidth=2,
    arrowcolor="#1f77b4",
    ax=40,
    ay=-40,
    font=dict(size=12, color="black")
)

# Show the figure
fig.show()

# =============================================================================
# TASK 2: Likelihood Formulation (Mathematical expressions displayed)
# =============================================================================

print("\n" + "="*60)
print("TASK 2: Likelihood Formulation")
print("="*60)

print("\nSingle measurement likelihood L(y_k | θ):")
print("L(y_k | θ) = (1 / (y_k * σ * sqrt(2π))) * exp(-(1/(2σ²)) * [ln(y_k) - ln(θ*K_nominal)]²)")

print("\nJoint likelihood for running history vector y⁽ᵏ⁾:")
print("L(y⁽ᵏ⁾ | θ) = ∏_{i=1}^{k} L(y_i | θ)")
print("             = ∏_{i=1}^{k} [1/(y_i*σ*sqrt(2π))] * exp(-(1/(2σ²)) * Σ_{i=1}^{k} [ln(y_i) - ln(θ*K_nominal)]²)")

print("\nLog-likelihood (simplified for computation):")
print("ℓ(y⁽ᵏ⁾ | θ) = constant - (k/2)*ln(σ²) - (1/(2σ²)) * Σ_{i=1}^{k} [ln(y_i) - ln(θ*K_nominal)]²")

# =============================================================================
# Additional visualization: Interactive slider to explore parameter effects
# =============================================================================

# Create a 3D surface plot showing how the prior changes with different parameters
alpha_range = np.linspace(2, 15, 20)
beta_range = np.linspace(1, 5, 20)
theta_grid = np.linspace(0.01, 1.0, 50)

# For demonstration, we'll create a subplot showing how the prior changes
fig2 = go.Figure()

# Add traces for different beta values while keeping alpha fixed
for beta_val in [1.5, 2.5, 4.0]:
    pdf_trace = beta.pdf(theta, alpha, beta_val)
    fig2.add_trace(go.Scatter(
        x=theta,
        y=pdf_trace,
        mode='lines',
        name=f'Beta(8, {beta_val})',
        line=dict(width=2)
    ))

fig2.update_layout(
    title=dict(
        text="Effect of Changing β on the Prior Distribution (α=8)",
        font=dict(size=18, family="Arial"),
        x=0.5
    ),
    xaxis=dict(title="θ", range=[0, 1.05]),
    yaxis=dict(title="Density"),
    template="plotly_white",
    width=900,
    height=450
)

fig2.show()

Expected prior stiffness efficiency E[Θ⁽⁰⁾] = 0.8421
Mode of the prior distribution = 0.9333



TASK 2: Likelihood Formulation

Single measurement likelihood L(y_k | θ):
L(y_k | θ) = (1 / (y_k * σ * sqrt(2π))) * exp(-(1/(2σ²)) * [ln(y_k) - ln(θ*K_nominal)]²)

Joint likelihood for running history vector y⁽ᵏ⁾:
L(y⁽ᵏ⁾ | θ) = ∏_{i=1}^{k} L(y_i | θ)
             = ∏_{i=1}^{k} [1/(y_i*σ*sqrt(2π))] * exp(-(1/(2σ²)) * Σ_{i=1}^{k} [ln(y_i) - ln(θ*K_nominal)]²)

Log-likelihood (simplified for computation):
ℓ(y⁽ᵏ⁾ | θ) = constant - (k/2)*ln(σ²) - (1/(2σ²)) * Σ_{i=1}^{k} [ln(y_i) - ln(θ*K_nominal)]²


##**Task 2: Structural Likelihood Formulation**

**1. Single Measurement Likelihood Contribution $L(y_k \mid \theta)$**

The physical measurement model is defined as:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \text{where } \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

Taking the natural logarithm of both sides:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Since $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, the logarithmic measurement $\ln(y_k)$ follows a Gaussian distribution:

$$\ln(y_k) \sim \mathcal{N}\left( \ln(\theta \cdot K_{\text{nominal}}),\, \sigma^2 \right)$$Applying the standard Log-Normal density function (or via change of variables $y_k = g(\epsilon_k)$), the single-point likelihood contribution $L(y_k \mid \theta)$ is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$$Alternatively written using logarithmic ratio rules:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{1}{2\sigma^2} \left[ \ln\left( \frac{y_k}{\theta \cdot K_{\text{nominal}}} \right) \right]^2 \right)$$

**2. Joint Likelihood Function for Running History $\mathbf{y}^{(k)}$**

Assuming measurement noise terms $\epsilon_1, \epsilon_2, \dots, \epsilon_k$ are independent and identically distributed (i.i.d.), the joint likelihood for the history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual likelihood contributions:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$

Substituting the single-point likelihood expression:

$$L(\mathbf{y}^{(k)} \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^{k} \frac{1}{y_i} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^{k} \left[ \ln(y_i) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2 \right)$$

##**Task 3: Mathematical Formulation of the Non-Conjugate Grid Update**

**1. Explanation of Non-Conjugacy**

An exact closed-form analytical expression for the posterior $f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist because the prior and likelihood families are non-conjugate:

*   The prior distribution is a Beta distribution over $\theta \in (0, 1]$, which features algebraic polynomial terms: $\theta^{\alpha-1} (1-\theta)^{\beta-1}$.

*   The likelihood $L(y_k \mid \theta)$ is a Log-Normal distribution with respect to $y_k$, which introduces $\theta$ inside a squared logarithmic term: $\exp\left(-\frac{1}{2\sigma^2} [\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})]^2 \right)$.

Multiplying these expressions yields a product of polynomial and complex transcendental terms $\theta^{\alpha-1} (1-\theta)^{\beta-1} \exp\left(-\frac{1}{2\sigma^2} [\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})]^2 \right)$.The marginal likelihood (evidence) normalizing constant $f_{\mathbf{Y}^{(k)}}(\mathbf{y}^{(k)}) = \int_{0}^{1} f_{\Theta}(\theta) L(\mathbf{y}^{(k)} \mid \theta) \, d\theta$ cannot be evaluated in closed form using standard functions, making exact analytical updating impossible.

**2. Recursive Posterior Relationship**

Exploiting the sequential Bayesian update property (where the posterior at step $k-1$ serves as the prior at step $k$), the recursive relationship up to a proportionality constant ($\propto$) is:$$f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta\vert{}\mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$Substituting the single-point log-normal likelihood:$$f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta\vert{}\mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$$

##**Task 4: Running Point Estimates**

**1. Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)**

The expected value of $\theta$ under the step-$k$ posterior distribution over the domain $(0, 1]$ is given by:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{y}^{(k)}] = \frac{\int_{0}^{1} \theta \cdot f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}$$If the posterior density function $f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is already normalized such that $\int_0^1 f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) d\theta = 1$, this simplifies to:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

**2. Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)**

The MAP estimate corresponds to the mode (location of maximum density) over the bounded domain $(0, 1]$:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta\vert{}\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$



##**Task 5: Algorithmic Grid Approximation and Normalization**

###**Step-by-Step Numerical Procedure**

**1.Grid Discretization & Boundary Handling:**

*   Define an $M$-point uniform spatial grid vector $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ spanning the physical interval $[0.01, 1.0]$.

*   Boundary Handling: To prevent non-physical singularities such as $\ln(0)$ or division by zero at $\theta = 0$, set the lower bound strictly above zero ($\theta_{\text{min}} = 0.01$). Similarly, clip/clamp any operations exceeding $\theta_{\text{max}} = 1.0$.

**2.Initialization ($k=0$):**

*   Compute the prior PDF values on the grid: $p_0(\theta_m) = \text{Beta}(\theta_m; 8, 1.5)$ for $m = 1, \dots, M$.

*   Normalize using the trapezoidal rule:$$C_0 = \text{Trapezoid}(p_0(\boldsymbol{\theta}), \boldsymbol{\theta}), \quad f_0(\boldsymbol{\theta}) = \frac{p_0(\boldsymbol{\theta})}{C_0}$$

**3.Sequential Step Update ($k = 1, 2, \dots, n$):**

*  Receive observation $y_k$.

*   Evaluate the unnormalized Gaussian likelihood vector in log-space on the grid:$$L(y_k \mid \theta_m) = \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta_m \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$$

*   Compute the unnormalized posterior grid:$$\tilde{f}_k(\theta_m) = f_{k-1}(\theta_m) \cdot L(y_k \mid \theta_m)$$

**4.Sequential Normalization via Trapezoidal Rule:**

*   Numerically integrate the unnormalized grid using np.trapezoid:$$C_k = \int_{0.01}^{1.0} \tilde{f}_k(\theta) \, d\theta \approx \text{np.trapezoid}(\tilde{f}_k(\boldsymbol{\theta}), \boldsymbol{\theta})$$

*   Update the normalized density array for the next iteration:$$f_k(\theta_m) = \frac{\tilde{f}_k(\theta_m)}{C_k}$$





##**Task 6: Performance Tracking and Degradation Convergence Analysis**

###**Python Simulation Code**

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta, norm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PARAMETERS
# ============================================================================

# True damage state
theta_true = 0.68

# Physical constants
K_nominal = 50.0  # kN/mm
sigma = 0.15      # log-space noise

# Prior parameters (Beta distribution)
alpha = 8
beta_param = 1.5

# Monitoring timeline
n_steps = 15

# Grid parameters
n_grid = 2000
theta_min = 1e-6  # Avoid singularity at 0
theta_max = 1.0
theta_grid = np.linspace(theta_min, theta_max, n_grid)
delta_theta = theta_grid[1] - theta_grid[0]

# ============================================================================
# STEP 1: Simulate Sensor Stream
# ============================================================================

np.random.seed(42)  # For reproducibility

# Generate noisy measurements from the log-normal model
epsilon = np.random.normal(0, sigma, n_steps)
y_measurements = theta_true * K_nominal * np.exp(epsilon)

print("Generated Sensor Readings:")
for k, y in enumerate(y_measurements, 1):
    print(f"  Step {k:2d}: y_k = {y:.3f} kN/mm")

# ============================================================================
# STEP 2: Initialize Grid and Prior
# ============================================================================

# Evaluate initial prior on the grid
prior_values = beta.pdf(theta_grid, alpha, beta_param)

# Store posterior densities for visualization at specific milestones
milestones = [0, 1, 2, 5, 10, 15]
posterior_history = {m: None for m in milestones}
posterior_history[0] = prior_values.copy()

# Storage for point estimates
bayes_estimates = np.zeros(n_steps + 1)
map_estimates = np.zeros(n_steps + 1)

# Initial estimates (prior only)
bayes_estimates[0] = np.trapz(theta_grid * prior_values, theta_grid)
map_estimates[0] = theta_grid[np.argmax(prior_values)]

# Current unnormalized posterior (starts as prior)
unnormalized_posterior = prior_values.copy()

# ============================================================================
# STEP 3: Sequential Bayesian Updating
# ============================================================================

print("\nSequential Bayesian Updates:")
print("-" * 60)

for k in range(1, n_steps + 1):
    y_k = y_measurements[k-1]

    # Compute likelihood at each grid point
    log_likelihood = -0.5 * ((np.log(y_k) - np.log(theta_grid * K_nominal)) / sigma) ** 2
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(log_likelihood)

    # Update unnormalized posterior
    unnormalized_posterior = unnormalized_posterior * likelihood

    # Handle numerical issues: clip extremely small values
    unnormalized_posterior = np.clip(unnormalized_posterior, 1e-300, None)

    # Normalize using trapezoidal rule
    Z = np.trapz(unnormalized_posterior, theta_grid)
    posterior = unnormalized_posterior / Z

    # Compute point estimates
    # Posterior Mean
    bayes_estimates[k] = np.trapz(theta_grid * posterior, theta_grid)

    # MAP (Maximum A Posteriori)
    map_estimates[k] = theta_grid[np.argmax(posterior)]

    # Store for milestones
    if k in milestones:
        posterior_history[k] = posterior.copy()

    # Print progress
    if k <= 5 or k % 5 == 0:
        print(f"Step {k:2d}: Bayes = {bayes_estimates[k]:.4f}, "
              f"MAP = {map_estimates[k]:.4f}, "
              f"True = {theta_true:.4f}")

print("-" * 60)

# ============================================================================
# PLOT 1: Posterior Density Evolution
# ============================================================================

fig1 = go.Figure()

# Define colors for milestones
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
labels = ['Prior (k=0)', 'k=1', 'k=2', 'k=5', 'k=10', 'k=15']

for i, k in enumerate(milestones):
    if posterior_history[k] is not None:
        fig1.add_trace(go.Scatter(
            x=theta_grid,
            y=posterior_history[k],
            mode='lines',
            name=labels[i],
            line=dict(color=colors[i], width=2.5 if k == 15 else 1.5),
            hovertemplate='θ = %{x:.3f}<br>Density = %{y:.3f}<extra></extra>'
        ))

# Add vertical line for true value
fig1.add_vline(
    x=theta_true,
    line_dash='dash',
    line_color='red',
    line_width=2,
    annotation_text=f'θ<sub>true</sub> = {theta_true}',
    annotation_position='top right'
)

fig1.update_layout(
    title=dict(
        text='Evolution of Posterior Density Over Time<br>'
             '<span style="font-size:14px;font-weight:normal;">'
             'Beta(8,1.5) Prior + Log-Normal Likelihood</span>',
        font=dict(size=20, family='Arial', color='black'),
        x=0.5
    ),
    xaxis=dict(
        title=dict(text='θ (Remaining Stiffness Efficiency)', font=dict(size=14)),
        range=[0, 1.0],
        tickvals=np.arange(0, 1.1, 0.1),
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title=dict(text='Posterior Density', font=dict(size=14)),
        gridcolor='lightgray'
    ),
    legend=dict(
        x=0.65,
        y=0.95,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1
    ),
    template='plotly_white',
    width=950,
    height=550,
    hovermode='x'
)

fig1.show()

# ============================================================================
# PLOT 2: Point Estimate Convergence
# ============================================================================

fig2 = go.Figure()

# Add Bayesian Mean trace
fig2.add_trace(go.Scatter(
    x=list(range(n_steps + 1)),
    y=bayes_estimates,
    mode='lines+markers',
    name='Bayesian Mean',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=8, symbol='circle'),
    hovertemplate='Step %{x}<br>Bayes Mean = %{y:.4f}<extra></extra>'
))

# Add MAP trace
fig2.add_trace(go.Scatter(
    x=list(range(n_steps + 1)),
    y=map_estimates,
    mode='lines+markers',
    name='MAP Estimate',
    line=dict(color='#ff7f0e', width=3, dash='dot'),
    marker=dict(size=8, symbol='square'),
    hovertemplate='Step %{x}<br>MAP = %{y:.4f}<extra></extra>'
))

# Add horizontal reference line at true value
fig2.add_hline(
    y=theta_true,
    line_dash='dash',
    line_color='red',
    line_width=2,
    annotation_text=f'θ<sub>true</sub> = {theta_true}',
    annotation_position='top right'
)

# Add shaded region for "convergence" (within 2% of true)
fig2.add_hrect(
    y0=theta_true * 0.98,
    y1=theta_true * 1.02,
    line_width=0,
    fillcolor='rgba(0, 255, 0, 0.1)',
    annotation_text='±2% Convergence Band',
    annotation_position='bottom right'
)

# Determine when convergence is achieved
convergence_step_bayes = None
convergence_step_map = None
tolerance = 0.02 * theta_true

for k in range(1, n_steps + 1):
    if convergence_step_bayes is None and abs(bayes_estimates[k] - theta_true) < tolerance:
        convergence_step_bayes = k
    if convergence_step_map is None and abs(map_estimates[k] - theta_true) < tolerance:
        convergence_step_map = k

# Mark convergence points
if convergence_step_bayes is not None:
    fig2.add_annotation(
        x=convergence_step_bayes,
        y=bayes_estimates[convergence_step_bayes],
        text=f'Bayes converges<br>at step {convergence_step_bayes}',
        showarrow=True,
        arrowhead=2,
        ax=-40,
        ay=-40,
        font=dict(size=10, color='#1f77b4')
    )

if convergence_step_map is not None:
    fig2.add_annotation(
        x=convergence_step_map,
        y=map_estimates[convergence_step_map],
        text=f'MAP converges<br>at step {convergence_step_map}',
        showarrow=True,
        arrowhead=2,
        ax=40,
        ay=-40,
        font=dict(size=10, color='#ff7f0e')
    )

fig2.update_layout(
    title=dict(
        text='Convergence of Point Estimates to True Damage State<br>'
             f'<span style="font-size:14px;font-weight:normal;">'
             f'θ<sub>true</sub> = {theta_true}, σ = {sigma}, K<sub>nominal</sub> = {K_nominal} kN/mm</span>',
        font=dict(size=20, family='Arial', color='black'),
        x=0.5
    ),
    xaxis=dict(
        title=dict(text='Inspection Step k', font=dict(size=14)),
        tickvals=list(range(0, n_steps + 1, 2)),
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title=dict(text='Estimated θ', font=dict(size=14)),
        range=[0.5, 0.9],
        tickvals=np.round(np.linspace(0.5, 0.9, 9), 2),
        gridcolor='lightgray'
    ),
    legend=dict(
        x=0.7,
        y=0.05,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1
    ),
    template='plotly_white',
    width=950,
    height=500,
    hovermode='x unified'
)

fig2.show()

# ============================================================================
# PLOT 3: Uncertainty Evolution (Credible Interval Width)
# ============================================================================

# Compute 95% credible intervals
ci_lower = np.zeros(n_steps + 1)
ci_upper = np.zeros(n_steps + 1)
ci_width = np.zeros(n_steps + 1)

for k in range(n_steps + 1):
    if k == 0:
        posterior_vals = prior_values
    else:
        posterior_vals = posterior_history[k] if k in posterior_history else None

    if posterior_vals is not None:
        # Compute cumulative distribution
        cdf = np.cumsum(posterior_vals * delta_theta)
        # Find 2.5% and 97.5% quantiles
        ci_lower[k] = theta_grid[np.searchsorted(cdf, 0.025)]
        ci_upper[k] = theta_grid[np.searchsorted(cdf, 0.975)]
        ci_width[k] = ci_upper[k] - ci_lower[k]

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=list(range(n_steps + 1)),
    y=ci_width,
    mode='lines+markers',
    name='95% CI Width',
    line=dict(color='#9467bd', width=3),
    marker=dict(size=8, symbol='diamond'),
    fill='tozeroy',
    fillcolor='rgba(148, 103, 189, 0.2)',
    hovertemplate='Step %{x}<br>CI Width = %{y:.4f}<extra></extra>'
))

# Add horizontal line showing convergence criterion
fig3.add_hline(
    y=0.02 * theta_true,
    line_dash='dash',
    line_color='green',
    line_width=2,
    annotation_text='Desired precision<br>(2% of θ<sub>true</sub>)',
    annotation_position='top right'
)

fig3.update_layout(
    title=dict(
        text='Uncertainty Reduction Over Time (95% Credible Interval Width)',
        font=dict(size=20, family='Arial', color='black'),
        x=0.5
    ),
    xaxis=dict(
        title=dict(text='Inspection Step k', font=dict(size=14)),
        tickvals=list(range(0, n_steps + 1, 2)),
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title=dict(text='95% Credible Interval Width', font=dict(size=14)),
        gridcolor='lightgray'
    ),
    template='plotly_white',
    width=950,
    height=450,
    hovermode='x'
)

fig3.show()

Generated Sensor Readings:
  Step  1: y_k = 36.630 kN/mm
  Step  2: y_k = 33.302 kN/mm
  Step  3: y_k = 37.469 kN/mm
  Step  4: y_k = 42.726 kN/mm
  Step  5: y_k = 32.827 kN/mm
  Step  6: y_k = 32.827 kN/mm
  Step  7: y_k = 43.088 kN/mm
  Step  8: y_k = 38.148 kN/mm
  Step  9: y_k = 31.688 kN/mm
  Step 10: y_k = 36.883 kN/mm
  Step 11: y_k = 31.717 kN/mm
  Step 12: y_k = 31.706 kN/mm
  Step 13: y_k = 35.257 kN/mm
  Step 14: y_k = 25.518 kN/mm
  Step 15: y_k = 26.249 kN/mm

Sequential Bayesian Updates:
------------------------------------------------------------
Step  1: Bayes = 0.8084, MAP = 0.8159, True = 0.6800
Step  2: Bayes = 0.7522, MAP = 0.7434, True = 0.6800
Step  3: Bayes = 0.7522, MAP = 0.7454, True = 0.6800
Step  4: Bayes = 0.7757, MAP = 0.7704, True = 0.6800
Step  5: Bayes = 0.7514, MAP = 0.7469, True = 0.6800
Step 10: Bayes = 0.7389, MAP = 0.7364, True = 0.6800
Step 15: Bayes = 0.6886, MAP = 0.6873, True = 0.6800
------------------------------------------------------------


**ANALYSIS SUMMARY**

1. Convergence Analysis:
   - Bayesian Mean converged to within 2% of θ_true at step 15
   - MAP estimate converged to within 2% of θ_true at step 15


2. Initial vs Final Estimates:
   - Initial prior mean: 0.8420
   - Final Bayesian mean: 0.6886
   - Final MAP estimate: 0.6873

3. Uncertainty Reduction:
   - Initial 95% CI width: 0.4202
   - Final 95% CI width: 0.1046
   - Reduction: 75.1%

4. Damage Detection Timeline:
   - The system confidently detected the 68% damage state after 15 readings
     (overcoming the optimistic Beta(8,1.5) prior)

5. Safety Implications:
   - The posterior distribution narrowed from width 0.4202 to 0.1046
   - This narrowing implies increasing confidence in the damage estimate
   - For structural safety, this allows:
     * Setting early warning thresholds
     * Estimating remaining useful life more accurately
     * Making timely maintenance decisions